# Dự báo doanh số 8 tuần cho phòng tài chính: chọn phương pháp bằng backtest

Bộ `store-sales` (Rossmann), tệp `train.csv`: doanh số ngày của 1 115 cửa hàng từ 01/01/2013 tới 31/07/2015
(1 017 209 dòng). Cột dùng: `Store`, `Date`, `Sales`, `Promo` (1 nếu ngày có khuyến mãi; lịch khuyến mãi do công
ty lên trước và chạy đồng loạt toàn chuỗi).

**Bối cảnh.** Phòng tài chính (FP&A) cần dự báo doanh số **theo tuần, 8 tuần tới** cho toàn chuỗi để lên kế hoạch
dòng tiền và nhân sự. Họ không cần model phức tạp; họ cần một con số tin được, biết nó thường sai bao nhiêu, và
hiểu nó dựa trên gì. Bạn:

1. dựng chuỗi tuần **so sánh được** (chỉ cửa hàng có dữ liệu mọi ngày);
2. viết ba phương pháp ở mức DA: seasonal naive (cùng tuần năm trước), moving average, và trung bình theo lịch
   khuyến mãi;
3. chọn phương pháp bằng **backtest** nhiều điểm cắt, không bằng cảm giác.

Track Data Science (module 07) đi sâu ARIMA, ETS, LightGBM và reconciliation; bài này dừng ở mức một DA cần để trả
lời FP&A một cách có trách nhiệm.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/intermediate/02-weekly-forecast-backtest -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc train.csv (khoảng 1 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

train = pd.read_csv(ensure("store-sales") / "train.csv", parse_dates=["Date"], dtype={"StateHoliday": str})
print(train.shape, train.Date.min().date(), "→", train.Date.max().date(), "|", train.Store.nunique(), "cửa hàng")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Chuỗi tuần so sánh được

- `comparable_stores(train) -> pd.Index`: các `Store` có dòng cho **mọi** ngày từ ngày nhỏ nhất tới ngày lớn nhất
  của `train` (ngày đóng cửa vẫn có dòng với `Open = 0`, nên thiếu dòng nghĩa là thiếu dữ liệu). Index tăng dần,
  tên `Store`.
- `weekly_sales(train, stores) -> pd.DataFrame`: chỉ các cửa hàng trong `stores`; tuần thứ Hai tới Chủ nhật, index
  `week` là ngày thứ Hai; chỉ giữ tuần **đủ 7 ngày** có dữ liệu. Cột `sales` (tổng `Sales`) và `promo` (`int`, 1 nếu
  tuần có ít nhất một ngày `Promo == 1`).

In [ ]:
def comparable_stores(train: pd.DataFrame) -> pd.Index:
    """Cửa hàng có dòng cho mọi ngày trong khoảng ngày của train."""
    # TODO
    raise NotImplementedError


def weekly_sales(train: pd.DataFrame, stores: pd.Index) -> pd.DataFrame:
    """Tuần thứ Hai - Chủ nhật, đủ 7 ngày: sales, promo; index week."""
    # TODO: thứ Hai của tuần = Date - dayofweek ngày
    raise NotImplementedError

In [ ]:
# explore: vì sao phải dùng cửa hàng so sánh được - tổng mọi cửa hàng vs tổng cửa hàng đủ ngày
stores = comparable_stores(train)
weekly = weekly_sales(train, stores)
everyone = weekly_sales(train, pd.Index(train.Store.unique()))
print(f"{len(stores)} cửa hàng đủ mọi ngày; {weekly.index.min().date()} → {weekly.index.max().date()}, {len(weekly)} tuần")
gap = train.groupby("Store").Date.nunique()
print("số ngày có dữ liệu của các cửa hàng còn lại:", gap[gap < gap.max()].value_counts().to_dict())
for name, w in [("mọi cửa hàng", everyone), ("so sánh được", weekly)]:
    jun, jul = w.loc["2014-06-02":"2014-06-23", "sales"].mean(), w.loc["2014-07-07":"2014-07-28", "sales"].mean()
    print(f"{name}: TB tuần 6/2014 {jun / 1e6:.1f}M → 7/2014 {jul / 1e6:.1f}M ({jul / jun - 1:+.1%})")
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(everyone.index, everyone.sales / 1e6, color="#e45756", lw=1, label="mọi cửa hàng (thiếu 180 cửa hàng 7-12/2014)")
ax.plot(weekly.index, weekly.sales / 1e6, color="#4c78a8", lw=1, label="cửa hàng so sánh được")
ax.set_ylabel("triệu"); ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, loc="lower left")
ax.set_title("Doanh số tuần xen kẽ cao / thấp theo lịch khuyến mãi; tổng mọi cửa hàng tụt giả từ 7/2014", loc="left")
plt.show()
print(weekly.groupby("promo").sales.agg(["size", "mean"]).assign(mean=lambda d: (d["mean"] / 1e6).round(2)))

## 2. Ba phương pháp dự báo ở mức DA

Mỗi hàm nhận `history` (Series doanh số tuần, theo thứ tự thời gian) và `h` (số tuần cần dự báo), trả `np.ndarray`
độ dài `h`:

- `seasonal_naive(history, h, m=52)`: tuần thứ i (i = 0..h−1) bằng giá trị ở vị trí `len(history) − m + (i % m)`,
  tức cùng tuần năm trước. `ValueError` nếu `history` ngắn hơn `m`.
- `moving_average(history, h, k=4)`: trung bình `k` tuần cuối, lặp lại `h` lần.
- `promo_average(history, h, promo, k=4)`: `promo` là Series cờ khuyến mãi theo tuần, phủ cả lịch sử lẫn tương lai
  (lịch khuyến mãi biết trước). Tuần tương lai thứ i là tuần thứ i trong `promo` có index **sau** tuần cuối của
  `history`; dự báo của nó là trung bình `k` tuần gần nhất trong `history` có **cùng cờ** khuyến mãi. `ValueError`
  nếu `promo` không có đủ `h` tuần tương lai.

In [ ]:
def seasonal_naive(history: pd.Series, h: int, m: int = 52) -> np.ndarray:
    """Cùng tuần m tuần trước, lặp lại nếu h > m."""
    # TODO
    raise NotImplementedError


def moving_average(history: pd.Series, h: int, k: int = 4) -> np.ndarray:
    """Trung bình k tuần cuối."""
    # TODO
    raise NotImplementedError


def promo_average(history: pd.Series, h: int, promo: pd.Series, k: int = 4) -> np.ndarray:
    """Trung bình k tuần gần nhất có cùng cờ khuyến mãi với tuần cần dự báo."""
    # TODO
    raise NotImplementedError

## 3. Backtest

`backtest(weekly, forecasters, cutoffs, h=8) -> pd.DataFrame`:

- `weekly` là kết quả `weekly_sales`; `forecasters` là dict tên → hàm `f(history, h, promo)` trả `h` giá trị;
- với mỗi `cutoff` (ngày thứ Hai, chuỗi), `history` là `sales` các tuần **trước** `cutoff`, thực tế là `h` tuần
  từ `cutoff`; thiếu tuần thực tế thì `ValueError`;
- mỗi cặp (cutoff, phương pháp) một dòng, theo thứ tự cutoff rồi thứ tự trong dict; cột `cutoff` (Timestamp),
  `method`, `mape` (trung bình `|dự báo − thực tế| / thực tế`), `bias` (`tổng(dự báo − thực tế) / tổng thực tế`).

`summarize(results) -> pd.DataFrame`: index `method`, cột `mape` (trung bình), `worst_mape` (lớn nhất), `bias`
(trung bình), sắp tăng dần theo `mape`.

In [ ]:
def backtest(weekly: pd.DataFrame, forecasters: dict, cutoffs: list, h: int = 8) -> pd.DataFrame:
    """Một dòng mỗi (cutoff, method): cutoff, method, mape, bias."""
    # TODO
    raise NotImplementedError


def summarize(results: pd.DataFrame) -> pd.DataFrame:
    """MAPE trung bình, MAPE tệ nhất, bias trung bình theo phương pháp; sắp theo mape."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: backtest 6 điểm cắt mỗi 4 tuần, thêm ETS (Holt-Winters trên log, mùa vụ 52 tuần) để so
from statsmodels.tsa.holtwinters import ExponentialSmoothing


def ets(history, h, promo):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = ExponentialSmoothing(np.log(history.to_numpy(dtype=float)), seasonal="add", seasonal_periods=52).fit()
    return np.exp(fit.forecast(h))


FORECASTERS = {
    "seasonal_naive": lambda history, h, promo: seasonal_naive(history, h),
    "moving_average": lambda history, h, promo: moving_average(history, h),
    "promo_average": lambda history, h, promo: promo_average(history, h, promo),
    "ets": ets,
}
CUTOFFS = ["2015-01-05", "2015-02-02", "2015-03-02", "2015-03-30", "2015-04-27", "2015-05-25"]  # ETS cần 2 năm lịch sử
results = backtest(weekly, FORECASTERS, CUTOFFS)
print(results.pivot(index="cutoff", columns="method", values="mape").round(3))
summarize(results).round(3)

In [ ]:
# explore: nhìn một điểm cắt - dự báo của từng phương pháp so với thực tế (cutoff 2015-03-02, có lễ Phục sinh)
cutoff = pd.Timestamp("2015-03-02")
history = weekly.loc[weekly.index < cutoff, "sales"]
actual = weekly.loc[weekly.index >= cutoff, "sales"].iloc[:8]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(history.index[-20:], history.iloc[-20:] / 1e6, color="#9aa5b1", label="lịch sử")
ax.plot(actual.index, actual / 1e6, color="black", marker="o", label="thực tế")
for (name, f), color in zip(FORECASTERS.items(), ["#4c78a8", "#f58518", "#54a24b", "#b279a2"]):
    ax.plot(actual.index, f(history, 8, weekly["promo"]) / 1e6, ls="--", color=color, label=name)
ax.set_ylabel("triệu"); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False, ncol=3, fontsize=8)
ax.set_title("Tuần lễ Phục sinh 2015 sớm hơn 2014 hai tuần: seasonal naive lệch pha", loc="left")
plt.show()

In [ ]:
# explore: dự báo 8 tuần sau dữ liệu cho FP&A, kèm khoảng từ sai số backtest
# Giả định: lịch khuyến mãi tiếp tục xen kẽ (tuần 20/07/2015 không khuyến mãi -> tuần 27/07 có).
future_weeks = pd.date_range(weekly.index[-1] + pd.Timedelta(weeks=1), periods=8, freq="7D")
future_promo = pd.Series([1 - (i % 2) for i in range(8)], index=future_weeks)
promo_all = pd.concat([weekly["promo"], future_promo])
summary = summarize(results)
best = summary.index[0]
point = FORECASTERS[best](weekly["sales"], 8, promo_all)
err = summary.loc[best, "mape"]
plan = pd.DataFrame({"promo": future_promo, "forecast_m": point / 1e6, "low_m": point * (1 - err) / 1e6,
                     "high_m": point * (1 + err) / 1e6}, index=future_weeks)
print("phương pháp:", best, f"| MAPE backtest {err:.1%}")
print(plan.round(1))
part = train[(train.Date >= "2015-07-27") & train.Store.isin(stores)].Sales.sum()
print(f"tuần 27/07 mới có 5 ngày dữ liệu (thứ Hai-thứ Sáu): {part / 1e6:.1f}M")

## 4. Đọc kết quả, rồi làm trắc nghiệm

1. Tổng doanh số tuần của **mọi** cửa hàng tụt bao nhiêu từ tháng 7/2014? Vì sao? Nếu dùng chuỗi đó, dự báo cho
   năm 2015 sẽ sai theo hướng nào?
2. Phương pháp nào có MAPE trung bình thấp nhất? Phương pháp nào tệ nhất, vì sao (nhìn đường doanh số xen kẽ)?
   ETS phức tạp hơn có thắng không?
3. Điểm cắt nào khó nhất cho mọi phương pháp, và vì sao (lịch, lễ)?
4. Viết cho FP&A ba dòng: con số 8 tuần, khoảng sai số thường gặp, và hai giả định quan trọng nhất (lịch khuyến
   mãi, cửa hàng so sánh được).

Xong thì mở app, làm phần **trắc nghiệm** của bài.